# MBPP Evaluation — Improved (Curriculum-Trained) Models

Evaluates the new curriculum-trained adapters from `New lora_outputs`
against MBPP using the same greedy pass@1 methodology as the baseline.

## What this notebook does
- Loads the **same MBPP test set** (257 problems) used in the baseline
- Evaluates each entry: base model + each training stage + final LoRA
- Saves full per-task JSON + summary CSV to `mbpp_improved_results\`
- Prints a score table at the end for direct comparison with baseline

## Models evaluated
| Model | Entries |
|-------|---------|
| 270m (improved) | base, s1_easy, s2_medium, final_lora |
| 1b (improved) | base, s1_easy, s2_medium, final_lora *(when available)* |
| 4b (improved) | base, s1_easy, s2_medium, final_lora *(when available)* |

In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import json as _json_mod
import re, subprocess, tempfile, time, csv, random, math, copy, ast as _ast
from collections import defaultdict
import numpy as np

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    GenerationConfig, set_seed as hf_set_seed,
)
from peft import PeftModel
from datasets import load_dataset

print(f"PyTorch:  {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
_sm = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
DTYPE = torch.bfloat16 if _sm >= 8 else torch.float16
print(f"Dtype:    {DTYPE}")

k:\anaconda3\envs\honor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch:  2.11.0.dev20260205+cu128  |  CUDA: True
GPU:      NVIDIA GeForce RTX 5070 Ti
VRAM:     15.9 GB
Dtype:    torch.bfloat16


In [4]:
HF_MODELS    = r"K:\Honor Project\hf_models"
NEW_LORA_DIR = r"K:\Honor Project\New lora_outputs"
RESULTS_DIR  = r"K:\Honor Project\mbpp_improved_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

MAX_NEW_TOKENS = 512
TEST_TIMEOUT   = 20
FORCE_RERUN    = False

# ── Per-model evaluation plans ────────────────────────────────────────────────

def _lora(size, stage):
    # Returns path to a stage adapter, or None if not found.
    # Checks direct folder first, then any checkpoint-* subfolder.
    base = os.path.join(NEW_LORA_DIR, f"gemma-3-{size}", stage)
    if not os.path.isdir(base):
        return None
    # Direct adapter (final_lora style)
    if os.path.exists(os.path.join(base, "adapter_config.json")):
        return base
    # Checkpoint subfolder — pick the highest step
    ckpts = []
    for name in os.listdir(base):
        full = os.path.join(base, name)
        if name.startswith("checkpoint-") and os.path.isdir(full):
            if os.path.exists(os.path.join(full, "adapter_config.json")):
                try:
                    ckpts.append((int(name.split("-")[1]), full))
                except ValueError:
                    pass
    if ckpts:
        return sorted(ckpts)[-1][1]  # highest step
    return None

MODEL_CONFIGS = []

for _size, _base_name in [
                           ("1b",   "gemma-3-1b-pt"),
                           ("4b",   "gemma-3-4b-pt")]:
    _base_path = os.path.join(HF_MODELS, _base_name)
    if not os.path.isdir(_base_path):
        print(f"  [SKIP] base not found: {_base_path}")
        continue

    _lora_root = os.path.join(NEW_LORA_DIR, f"gemma-3-{_size}")
    if not os.path.isdir(_lora_root):
        print(f"  [SKIP] no adapters found for {_size}")
        continue

    _plan = [("base", 0, None)]
    for _stage, _step in [("s1", 800), ("s2", 1600), ("s3", 2400)]:
        _p = _lora(_size, _stage)
        if _p:
            _label = {"s1": "s1_easy", "s2": "s2_medium", "s3": "s3_hard"}[_stage]
            _plan.append((_label, _step, _p))
    _fl = _lora(_size, "final_lora")
    if _fl:
        _plan.append(("final_lora", _step if _plan else 0, _fl))

    MODEL_CONFIGS.append({
        "size":      f"{_size}_improved",
        "base_path": _base_path,
        "plan":      _plan,
    })

print(f"RESULTS_DIR  = {RESULTS_DIR}")
print(f"FORCE_RERUN  = {FORCE_RERUN}")
print()
for cfg in MODEL_CONFIGS:
    print(f"  {cfg['size']}: {[l for l,_,_ in cfg['plan']]}")

  [SKIP] no adapters found for 4b
RESULTS_DIR  = K:\Honor Project\mbpp_improved_results
FORCE_RERUN  = False

  1b_improved: ['base', 's1_easy', 's2_medium', 's3_hard', 'final_lora']


In [5]:
print("Loading MBPP (google-research-datasets/mbpp, sanitized, test)...")
try:
    _ds = load_dataset("google-research-datasets/mbpp", "sanitized", split="test")
except Exception as _e:
    print(f"  Fallback: {_e}")
    _ds = load_dataset("mbpp", split="test")

mbpp_problems = []
for _row in _ds:
    _r = dict(_row)
    if "text" not in _r:
        _r["text"] = _r.get("prompt", "")
    if "test_setup_code" not in _r:
        _raw = _r.get("test_imports", "[]") or "[]"
        _imports = _json_mod.loads(_raw) if isinstance(_raw, str) else (_raw or [])
        _r["test_setup_code"] = "\n".join(_imports)
    if isinstance(_r.get("test_list"), str):
        _r["test_list"] = _json_mod.loads(_r["test_list"])
    mbpp_problems.append(_r)

print(f"  {len(mbpp_problems)} problems loaded.")

Loading MBPP (google-research-datasets/mbpp, sanitized, test)...
  257 problems loaded.


In [6]:
# ── Seeding ───────────────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    hf_set_seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ── Code extraction ───────────────────────────────────────────────────────────
def strip_fences(text):
    m = re.search(r"```(?:python)?\s*(.*?)\s*```", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()

def extract_func_name(test_list):
    for t in test_list:
        m = re.match(r"\s*assert\s+(\w+)\s*\(", t.strip())
        if m:
            return m.group(1)
    return None

def extract_function(text, func_name=None):
    text = strip_fences(text)
    if func_name:
        pat = rf"(^|\n)(def\s+{re.escape(func_name)}\s*\([\s\S]*?)(?=\n\s*def\s+|\n\s*class\s+|\n\s*if\s+__name__|\Z)"
        m = re.search(pat, text)
        if m:
            return m.group(2).rstrip()
    m2 = re.search(r"(^|\n)(def\s+\w+\s*\([\s\S]*?)(?=\n\s*def\s+|\n\s*class\s+|\n\s*if\s+__name__|\Z)", text)
    return m2.group(2).rstrip() if m2 else text.rstrip()

# ── Per-test runner ───────────────────────────────────────────────────────────
def run_tests_detailed(raw_gen, prob):
    tests  = prob["test_list"]
    setup  = prob.get("test_setup_code", "") or ""
    fname  = extract_func_name(tests)
    code   = extract_function(raw_gen, fname)
    try:
        _ast.parse(code)
        syntax_valid = True
    except SyntaxError:
        syntax_valid = False
    if not syntax_valid:
        return {"syntax_valid": False, "tests_passed": 0,
                "tests_total": len(tests), "passed_all": False,
                "error_type": "SyntaxError", "error_msg": "ast.parse failed"}
    header = (setup.strip() + "\n\n" if setup.strip() else "") + code + "\n\n"
    tests_passed = 0
    first_error_type = "OK"
    first_error_msg  = ""
    for assert_line in tests:
        src_code = header + assert_line + "\n"
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py",
                                         delete=False, encoding="utf-8") as f:
            f.write(src_code)
            tmp = f.name
        try:
            r = subprocess.run(["python", tmp], capture_output=True,
                               text=True, timeout=TEST_TIMEOUT)
            if r.returncode == 0:
                tests_passed += 1
            else:
                err_out = (r.stderr or r.stdout or "").strip()
                if first_error_type == "OK":
                    if "AssertionError" in err_out:
                        first_error_type = "AssertionError"
                    elif "SyntaxError" in err_out:
                        first_error_type = "SyntaxError"
                    else:
                        first_error_type = "RuntimeError"
                    first_error_msg = err_out[:200]
        except subprocess.TimeoutExpired:
            if first_error_type == "OK":
                first_error_type = "Timeout"
                first_error_msg  = f"Timed out after {TEST_TIMEOUT}s"
        except Exception as exc:
            if first_error_type == "OK":
                first_error_type = "RuntimeError"
                first_error_msg  = str(exc)[:200]
        finally:
            try:
                os.remove(tmp)
            except Exception:
                pass
    return {
        "syntax_valid": syntax_valid,
        "tests_passed": tests_passed,
        "tests_total":  len(tests),
        "passed_all":   tests_passed == len(tests),
        "error_type":   first_error_type if tests_passed < len(tests) else "OK",
        "error_msg":    first_error_msg,
    }

def build_input(prob, tok):
    text, tests = prob["text"], prob["test_list"]
    instr = (
        "Write a Python function to solve the following problem.\n"
        "Output only the Python function - no explanation, no markdown fences.\n\n"
        f"Problem: {text}\n\n"
        "The function must pass these tests:\n" + "\n".join(tests)
    )
    return tok(f"### User:\n{instr}\n### Assistant:\n", return_tensors="pt")

def make_greedy_cfg(eos_id):
    return GenerationConfig(
        do_sample=False, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=eos_id
    )

def evaluate_model(model, tok, problems, label):
    set_seed(42)
    gen_cfg  = make_greedy_cfg(tok.eos_token_id)
    per_task = {}
    t0       = time.time()
    for prob in tqdm(problems, desc=label, leave=False):
        tid     = prob["task_id"]
        inp     = build_input(prob, tok)
        inp     = {k: v.to(model.device) for k, v in inp.items()}
        inp_len = inp["input_ids"].shape[1]
        with torch.inference_mode():
            out = model.generate(**inp, generation_config=gen_cfg)
        full_gen = tok.decode(out[0][inp_len:], skip_special_tokens=True)
        result   = run_tests_detailed(full_gen, prob)
        result["code"] = full_gen
        per_task[str(tid)] = result
    n_pass  = sum(1 for v in per_task.values() if v["passed_all"])
    total   = len(per_task)
    score   = n_pass / total * 100
    elapsed = time.time() - t0
    return score, n_pass, total, elapsed, per_task

print("Evaluation functions defined.")

Evaluation functions defined.


In [7]:
eval_summary = []

for cfg in MODEL_CONFIGS:
    size      = cfg["size"]
    base_path = cfg["base_path"]
    plan      = cfg["plan"]

    print(f"\n{'='*70}")
    print(f"Model: gemma-3-{size}  |  {len(plan)} entries")
    print(f"  plan: {[l for l,_,_ in plan]}")
    print(f"{'='*70}")

    print(f"  Loading base on CPU...")
    _tok = AutoTokenizer.from_pretrained(base_path, local_files_only=True)
    if _tok.pad_token is None:
        _tok.pad_token = _tok.eos_token
    _tok.padding_side = "right"
    _base_model = AutoModelForCausalLM.from_pretrained(
        base_path, local_files_only=True,
        device_map=None, torch_dtype=DTYPE, attn_implementation="sdpa",
    )
    _base_model.config.use_cache = False
    _base_model.eval()
    print("  Base loaded.")

    for label, step, lora_path in plan:
        out_json = os.path.join(RESULTS_DIR, f"{size}_{label}.json")
        out_csv  = os.path.join(RESULTS_DIR, f"{size}_{label}.csv")

        if not FORCE_RERUN and os.path.exists(out_json) and os.path.exists(out_csv):
            with open(out_csv, encoding="utf-8") as f_:
                row = list(csv.DictReader(f_))[0]
            print(f"  [cached] {size}/{label:<18}  {float(row['pass_rate']):.2f}%  ({row['passed']}/{row['total']})")
            eval_summary.append({
                "size": size, "label": label, "step": step,
                "pass_rate": float(row["pass_rate"]),
                "passed": int(row["passed"]), "total": int(row["total"]),
                "runtime_sec": float(row.get("runtime_sec", 0)),
            })
            continue

        if lora_path is None:
            print(f"\n  Evaluating base ({size})...")
            if torch.cuda.is_available():
                _base_model = _base_model.to("cuda")
            _model   = _base_model
            _is_base = True
        elif os.path.isdir(lora_path):
            print(f"\n  Merging LoRA: {os.path.relpath(lora_path, NEW_LORA_DIR)}...")
            _base_copy = copy.deepcopy(_base_model)
            _model = PeftModel.from_pretrained(_base_copy, lora_path)
            _model = _model.merge_and_unload()
            del _base_copy
            if torch.cuda.is_available():
                _model = _model.to("cuda")
            _is_base = False
        else:
            print(f"  [SKIP] LoRA not found: {lora_path}")
            continue

        _model.eval()
        score, n_pass, total, elapsed, per_task = evaluate_model(
            _model, _tok, mbpp_problems, label=f"{size}/{label}"
        )
        print(f"  {size}/{label:<18}  {score:.2f}%  ({n_pass}/{total})  {elapsed/60:.1f} min")

        with open(out_json, "w", encoding="utf-8") as f_:
            _json_mod.dump(per_task, f_, indent=2, ensure_ascii=False)

        row = {
            "size": size, "label": label, "step": step,
            "pass_rate": round(score, 4), "passed": n_pass, "total": total,
            "runtime_sec": round(elapsed, 1),
        }
        with open(out_csv, "w", newline="", encoding="utf-8") as f_:
            w = csv.DictWriter(f_, fieldnames=list(row.keys()))
            w.writeheader(); w.writerow(row)

        eval_summary.append(row)

        if _is_base:
            _base_model = _base_model.cpu()
            torch.cuda.empty_cache()
        else:
            del _model
            torch.cuda.empty_cache()

    del _base_model, _tok
    torch.cuda.empty_cache()
    print(f"\n  Done with {size}.")

print(f"\n{'='*60}")
print("EVALUATION SUMMARY")
print(f"{'='*60}")
print(f"  {'size':<20} {'label':<20} {'step':>6}  {'pass@1(%)':>10}  {'passed':>8}")
print(f"  {'-'*68}")
for r in eval_summary:
    print(f"  {r['size']:<20} {r['label']:<20} {r['step']:>6}  {r['pass_rate']:>10.2f}  {r['passed']:>8}/{r['total']}")
print(f"\nAll results saved to: {RESULTS_DIR}")


Model: gemma-3-1b_improved  |  5 entries
  plan: ['base', 's1_easy', 's2_medium', 's3_hard', 'final_lora']
  Loading base on CPU...


`torch_dtype` is deprecated! Use `dtype` instead!


  Base loaded.

  Evaluating base (1b_improved)...


1b_improved/base:   0%|          | 0/257 [00:00<?, ?it/s]`generation_config` default values have been modified to match model-specific defaults: {'do_sample': True, 'top_k': 64, 'top_p': 0.95, 'bos_token_id': 2, 'eos_token_id': [1, 106]}. If this is not desired, please set these values explicitly.


  1b_improved/base                0.78%  (2/257)  51.7 min

  Merging LoRA: gemma-3-1b\s1\checkpoint-3000...


  1b_improved/s1_easy             8.17%  (21/257)  36.4 min

  Merging LoRA: gemma-3-1b\s2\checkpoint-3000...


  1b_improved/s2_medium           5.06%  (13/257)  34.7 min

  Merging LoRA: gemma-3-1b\s3\checkpoint-3000...


  1b_improved/s3_hard             6.23%  (16/257)  36.9 min

  Merging LoRA: gemma-3-1b\final_lora...


  1b_improved/final_lora          6.23%  (16/257)  36.5 min

  Done with 1b_improved.

EVALUATION SUMMARY
  size                 label                  step   pass@1(%)    passed
  --------------------------------------------------------------------
  1b_improved          base                      0        0.78         2/257
  1b_improved          s1_easy                 800        8.17        21/257
  1b_improved          s2_medium              1600        5.06        13/257
  1b_improved          s3_hard                2400        6.23        16/257
  1b_improved          final_lora             2400        6.23        16/257

All results saved to: K:\Honor Project\mbpp_improved_results
